In [0]:
from pyspark.sql import functions as F


source_df = spark.read.option("header", True).option("inferSchema", True).csv("file:/Workspace/Users/kevin.wise@thermofisher.com/inventory_rebalance/data_inputs/source_data.csv")
destination_df = spark.read.option("header", True).option("inferSchema", True).csv("file:/Workspace/Users/kevin.wise@thermofisher.com/inventory_rebalance/data_inputs/destination_data.csv")
lanes_df = spark.read.option("header", True).option("inferSchema", True).csv("file:/Workspace/Users/kevin.wise@thermofisher.com/inventory_rebalance/data_inputs/Active Supply Lanes.csv")


source_df = (
    source_df
    .withColumn("SKU_Number", F.trim(F.col("SKU_Number")))
    .withColumn("SKU_Source", F.upper(F.trim(F.col("SKU_Source"))))
    .withColumn("Branch_Code", F.upper(F.trim(F.col("Branch_Code"))))
)

destination_df = (
    destination_df
    .withColumn("SKU_Number", F.trim(F.col("SKU_Number")))
    .withColumn("SKU_Source", F.upper(F.trim(F.col("SKU_Source"))))
    .withColumn("Branch_Code", F.upper(F.trim(F.col("Branch_Code"))))
)


lanes_df = (
    lanes_df
    .select(
        F.upper(F.trim(F.col("From Branch"))).alias("from_branch"),
        F.upper(F.trim(F.col("To Branch"))).alias("to_branch"),
        F.to_date(F.col("Effective From")).alias("effective_start_date"),
        F.to_date(F.col("Effective Thru")).alias("effective_end_date")
    )
    .dropna(subset=["from_branch", "to_branch", "effective_start_date"])
    .dropDuplicates()
)